# MetaARIMA: Meta-Learning for ARIMA Order Selection

MetaARIMA is a two-stage method that uses meta-learning to speed up ARIMA configuration selection:

1. **Meta-training** (offline, once per frequency): train a meta-learner that maps time-series features to promising ARIMA configurations.
2. **Inference** (online, per series): extract features, shortlist configurations via the meta-learner, and select the best by AICc (using successive halving).

This notebook demonstrates both stages on M4 monthly data:
- **Part 1** — Training MetaARIMA from pre-computed metadata
- **Part 2** — Loading a trained model and forecasting a new series

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# !pip install metaforecast catboost

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

from metaforecast.coseal import MetaARIMA
from metaforecast.coseal.metaarima._base import (
    CATBOOST_PARAMS,
    ORDER_MAX,
    MetaARIMAUtils,
)

## Part 1 — Training MetaARIMA

Training requires two pre-computed datasets (stored under `assets/metadata/metaarima/`):

- **Features** (`features-dev,m4_monthly.csv`): time-series features (via `tsfeatures`) for each series in the M4 monthly corpus.
- **Scores** (`arima-dev,m4_monthly.csv`): per-configuration error scores (e.g. MASE) obtained by cross-validating every ARIMA(p,d,q)(P,D,Q)[12] on each series.

The meta-learner maps features → configuration quality, so that at inference time we only need to fit a small shortlist instead of all 400 configurations.

### 1.1 Load metadata

In [5]:
METADATA_DIR = Path("../assets/metadata/metaarima")

features = pd.read_csv(METADATA_DIR / "features-dev,m4_monthly.csv")
scores = pd.read_csv(METADATA_DIR / "arima-dev,m4_monthly.csv")

print(f"Features: {features.shape}")
print(f"Scores:   {scores.shape}")
features.head()

Features: (48000, 43)
Scores:   (47990, 422)


,unique_id,hurst,series_length,unitroot_pp,unitroot_kpss,hw_alpha,hw_beta,hw_gamma,stability,nperiods,...,entropy,crossing_points,arch_lm,x_acf1,x_acf10,diff1_acf1,diff1_acf10,diff2_acf1,diff2_acf10,seas_acf1
0,T000000,0.985268,451,-39.391553,0.932401,0.468393,4.030205e-14,2.681482e-01,0.741432,1,...,0.460897,70,0.780341,0.888239,4.598817,0.069424,0.440084,-0.410274,0.493051,0.824569
1,T000001,0.908363,451,-97.631799,0.498813,0.388442,5.947693e-14,1.750788e-11,0.720775,1,...,0.623007,99,0.396654,0.750923,4.009390,-0.335965,0.182563,-0.573050,0.369051,0.594744
2,T000002,1.048263,451,1.419588,7.326279,0.765415,8.589195e-03,0.000000e+00,0.960555,1,...,0.322827,5,0.991499,0.990988,9.135706,-0.094058,0.088565,-0.488078,0.318407,0.905644
3,T000003,0.957351,64,-10.057401,1.110518,1.000000,2.980214e-14,0.000000e+00,0.838862,1,...,0.425187,10,0.616788,0.843726,3.317484,-0.090165,0.113868,-0.565203,0.427756,0.353844
4,T000004,0.806031,94,-24.613706,0.483806,0.547893,4.753660e-13,2.470836e-13,0.714338,1,...,0.700600,25,0.395925,0.707535,1.496378,-0.395666,0.288395,-0.658733,0.630672,0.067262


In [6]:
scores.head()

,"ARIMA(0,0,0)(0,0,0)[12]","ARIMA(0,0,0)(0,0,1)[12]","ARIMA(0,0,0)(0,1,0)[12]","ARIMA(0,0,0)(0,1,1)[12]","ARIMA(0,0,0)(1,0,0)[12]","ARIMA(0,0,0)(1,0,1)[12]","ARIMA(0,0,0)(1,1,0)[12]","ARIMA(0,0,0)(1,1,1)[12]","ARIMA(0,0,1)(0,0,0)[12]","ARIMA(0,0,1)(0,0,1)[12]",...,zero_mean,constant_variance,normality,no_autocorrelation,best_config,unique_id,coef_intercept,coef_ar4,coef_ma3,coef_ma4
0,1.771019,2.107085,1.446673,1.285196,1.729448,2.151250,1.300180,1.400922,1.786102,2.070872,...,0.298118,0.069565,0.009970,0.168597,"ARIMA(3,1,2)(1,1,1)[12]",T000003,NaN,NaN,NaN,NaN
1,0.860998,0.815904,0.833570,0.823626,0.847664,0.778967,0.785390,0.913463,0.843784,0.759593,...,0.625954,0.476135,0.613101,0.977522,"ARIMA(3,0,0)(0,0,1)[12]",T000004,4357.338689,NaN,NaN,NaN
2,2.071185,1.588594,0.719766,0.714942,1.280673,1.548584,0.718571,0.762275,2.017743,1.529774,...,0.196821,0.289749,0.073341,0.822858,"ARIMA(4,0,2)(0,1,0)[12]",T000006,NaN,-0.471172,NaN,NaN
3,3.146555,2.307692,0.999853,0.946378,1.499497,0.000000,0.951541,1.181904,3.051803,2.186121,...,NaN,NaN,NaN,NaN,"ARIMA(0,0,0)(1,0,1)[12]",T000007,7065.759451,NaN,NaN,NaN
4,1.201838,1.085918,2.968126,2.884460,2.494937,2.297044,2.847976,2.857732,1.136325,1.002673,...,0.806808,0.000057,0.004725,0.164530,"ARIMA(0,1,1)(1,0,0)[12]",T000000,NaN,NaN,NaN,NaN


In [7]:
# Merge features and scores on unique_id
id_col = "unique_id"
merged = scores.merge(features, on=id_col).set_index(id_col)

# Separate into X (features) and Y (per-config scores)
SEASON_LENGTH = 12

model_names = MetaARIMAUtils.get_models_sf(
    season_length=SEASON_LENGTH, max_config=ORDER_MAX, return_names=True
)

feature_cols = features.set_index(id_col).columns.tolist()
X = merged[feature_cols].fillna(-1)
Y = merged[[c for c in model_names if c in merged.columns]]

print(f"X (features):       {X.shape}")
print(f"Y (config scores):  {Y.shape}")
print(f"Configurations:     {len(model_names)}")

X (features):       (47990, 42)
Y (config scores):  (47990, 400)
Configurations:     400


### 1.2 Create and train MetaARIMA

Key parameters:
- **`n_trials=25`** — shortlist 25 configurations per series at inference time.
- **`mmr_lambda=0.75`** — balance relevance (predicted score) and diversity (low correlation) in the shortlist. `1` = pure relevance, `0` = pure diversity.
- **`base_optim='halving'`** — use successive halving (instead of exhaustive search) to select the final ARIMA from the shortlist.

In [9]:
catboost_params = CATBOOST_PARAMS["monthly"]

meta_arima = MetaARIMA(
    model=CatBoostRegressor(**catboost_params),
    freq="ME",
    season_length=SEASON_LENGTH,
    n_trials=25,
    quantile_thr=0.5,
    pca_n_components=100,
    mmr_lambda=0.75,
    base_optim="halving",
)

meta_arima.meta_fit(X, Y)

print(f"Meta-learner trained on {X.shape[0]} series.")
print(f"Number of target configurations: {len(meta_arima.model_names)}")

Meta-learner trained on 47990 series.
Number of target configurations: 400


### 1.3 Save the trained model

In [ ]:
# OUTPUT_DIR = Path("../assets/pretrained/metaarima")
OUTPUT_DIR = Path("./metaarima_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "m4_monthly.joblib.gz"
meta_arima.save(str(output_path))

print(f"Saved to {output_path}")
print(f"File size: {output_path.stat().st_size / 1e6:.1f} MB")

---

## Part 2 — Inference on a New Series

Once trained (or loaded from disk), MetaARIMA acts like a regular forecasting model:
1. `.fit(df, freq, seas_length)` — extract features, shortlist configurations, select the best.
2. `.predict(h)` — forecast *h* steps ahead.

### 2.1 Load a pre-trained model

In [ ]:
meta = MetaARIMA.load(str(OUTPUT_DIR / "m4_monthly.joblib.gz"))

print(f"Loaded model: freq={meta.freq}, season_length={meta.season_length}")
print(f"Meta-learner fitted: {meta.is_fit}")
print(f"Shortlist size (n_trials): {meta.n_trials}")

### 2.2 Create an example time series

In [ ]:
np.random.seed(42)

n = 240  # 20 years of monthly data
t = np.arange(n)

# Trend + seasonality + noise
trend = 0.02 * t
seasonal = 3 * np.sin(2 * np.pi * t / 12)
noise = np.random.normal(0, 0.5, n)
y_values = 50 + trend + seasonal + noise

df = pd.DataFrame({
    "unique_id": ["example_series"] * n,
    "ds": pd.date_range("2005-01-01", periods=n, freq="ME"),
    "y": y_values,
})

df.tail()

In [ ]:
df.set_index("ds")["y"].plot(
    figsize=(12, 3), title="Example monthly series", ylabel="y"
);

### 2.3 Fit and forecast

In [ ]:
meta.fit(df, freq="ME", seas_length=12)

print(f"Selected configuration: {meta.selected_config}")

In [ ]:
# Point forecast
forecast = meta.predict(h=18)

forecast.head()

In [ ]:
# Forecast with 95% prediction intervals
forecast_ci = meta.predict(h=18, level=[95])

forecast_ci.head()

### 2.4 Visualise

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))

# Plot the last 60 observations + forecast
history = df.tail(60).set_index("ds")
ax.plot(history.index, history["y"], label="History", color="black")

fc = forecast_ci.set_index("ds")
ax.plot(fc.index, fc["MetaARIMA"], label="MetaARIMA forecast", color="tab:blue")
ax.fill_between(
    fc.index,
    fc["MetaARIMA-lo-95"],
    fc["MetaARIMA-hi-95"],
    alpha=0.2,
    color="tab:blue",
    label="95% interval",
)

ax.set_title(f"MetaARIMA forecast — selected config: {meta.selected_config}")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()

### 2.5 Inspect the configuration shortlist

We can also look at the shortlist of ARIMA configurations that the meta-learner proposed before successive halving selected the final one.

In [18]:
from metaforecast.coseal.metaarima._base import tsfeatures_uid

# Extract features for the example series
feat_df = tsfeatures_uid(df, freq=12)

print(f"Feature vector shape: {feat_df.shape}")
feat_df.T.head(10)

Feature vector shape: (1, 42)


,example_series
hurst,8.442024e-01
series_length,2.400000e+02
unitroot_pp,-6.202842e+01
unitroot_kpss,2.085658e+00
hw_alpha,1.490116e-08
hw_beta,7.844652e-09
hw_gamma,0.000000e+00
stability,3.357124e-01
nperiods,1.000000e+00
seasonal_period,1.200000e+01


In [19]:
# Get the configuration shortlist
shortlist = meta.meta_predict(feat_df)[0]

print(f"Shortlisted {len(shortlist)} configurations (n_trials={meta.n_trials}):")
for i, cfg in enumerate(shortlist, 1):
    marker = " <-- selected" if cfg == meta.selected_config else ""
    print(f"  {i:2d}. {cfg}{marker}")

Shortlisted 25 configurations (n_trials=25):
   1. ARIMA(1,0,2)(1,1,0)[12]
   2. ARIMA(2,1,2)(0,1,1)[12]
   3. ARIMA(1,0,4)(1,0,1)[12]
   4. ARIMA(1,0,2)(1,1,1)[12]
   5. ARIMA(2,0,1)(1,1,0)[12]
   6. ARIMA(4,0,0)(0,1,1)[12]
   7. ARIMA(4,0,0)(1,1,0)[12]
   8. ARIMA(2,1,4)(1,1,1)[12]
   9. ARIMA(1,0,4)(1,1,0)[12]
  10. ARIMA(3,0,1)(1,1,0)[12]
  11. ARIMA(1,0,3)(1,1,0)[12]
  12. ARIMA(2,0,4)(1,1,0)[12]
  13. ARIMA(0,1,4)(1,1,1)[12]
  14. ARIMA(4,1,3)(0,1,1)[12]
  15. ARIMA(3,1,3)(0,1,1)[12]
  16. ARIMA(4,0,1)(1,1,0)[12]
  17. ARIMA(1,0,1)(1,1,0)[12]
  18. ARIMA(2,0,3)(1,1,0)[12]
  19. ARIMA(2,1,2)(1,1,1)[12] <-- selected
  20. ARIMA(1,0,2)(0,1,1)[12]
  21. ARIMA(4,1,4)(1,1,1)[12]
  22. ARIMA(4,1,2)(1,1,1)[12]
  23. ARIMA(2,0,2)(1,0,1)[12]
  24. ARIMA(2,0,2)(1,1,0)[12]
  25. ARIMA(3,1,3)(1,1,1)[12]


---

## Summary

| Step | Method | Description |
|------|--------|-------------|
| Meta-train | `meta_arima.meta_fit(X, Y)` | Train the meta-learner from features + error scores |
| Save | `meta_arima.save(path)` | Serialise to disk |
| Load | `MetaARIMA.load(freq=)` or `MetaARIMA.load(path=)` | Restore a bundled or custom model |
| Fit | `meta.fit(df, freq, seas_length)` | Shortlist + select best ARIMA for a new series |
| Predict | `meta.predict(h, level)` | Forecast with optional prediction intervals |